# Biblical Llama 3.1 70B Fine-Tuning with Unsloth (4-bit QLoRA v2)

**Base Model:** Llama 3.1 70B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — serves 4-bit model)

**Key Difference from v1:** This version uses 4-bit QLoRA training and saves directly to 4-bit quantized format, avoiding post-training quantization issues.

**Chat Template:** `<|begin_of_text|><|start_header_id|>role<|end_header_id|>content<|eot_id|>`

## Step 1: Configuration

All paths and variables for easy configuration.

In [3]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================
# Training: DGX Spark (128GB unified memory)
# Deployment: A5000 GPU server (24GB VRAM) via vLLM (4-bit model fits in ~40GB)

# =========================== MODEL CONFIGURATION ===========================
# Base model to fine-tune (use Unsloth's pre-quantized 4-bit model for faster loading)
BASE_LLM = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"

# Name for output folders and model identification
MODEL_NAME_BASE = "biblical_llama31_70b_instruct_unsloth_4bit"

# =========================== INPUT DATA ===========================
# Path to training data (Augmentoolkit output)
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset"

# =========================== OUTPUT DIRECTORIES ===========================
# All outputs organized under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"       # LoRA weights during training
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model_4bit"    # 4-bit merged model for vLLM

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048    # Max tokens per example
BATCH_SIZE = 1           # Per-device batch size (70B needs lower batch size)
GRAD_ACCUM = 8           # Gradient accumulation (effective batch = 8)
LEARNING_RATE = 1e-4     # Lower LR for 70B (larger models need gentler updates)
TARGET_EPOCHS = 1        # Number of training epochs

# =========================== LoRA CONFIGURATION ===========================
# LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training
# small adapter matrices instead of full model weights.
#
# For 70B 4-bit QLoRA, we use lower rank to manage VRAM since the base
# model is already highly capable and needs less adaptation.
#
# LORA_R (Rank): Controls adapter capacity
#   - Lower (4-8): Less aggressive, preserves base model behavior
#   - Higher (16-64): More capacity for new knowledge, risk of overfitting
#   - Recommendation: 8 for 70B QLoRA (model is already very capable)
#
# LORA_ALPHA: Scaling factor for LoRA weights
#   - Typically set to 2x LORA_R (e.g., r=8 → alpha=16)
#   - Higher alpha = stronger influence of fine-tuning
#
# LORA_DROPOUT: Regularization during training
#   - 0: No dropout (faster training)
#   - 0.05-0.1: Mild regularization for larger datasets
#
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# =========================== LoRA TARGET MODULES ===========================
# Which layers to fine-tune. Llama architecture has these trainable layers:
#
# ATTENTION MODULES (Recommended for persona/style transfer)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ q_proj       │ Query projection        │ "What am I looking for?"            │
# │              │                         │ Core attention steering             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ k_proj       │ Key projection          │ "What info do I have?"              │
# │              │                         │ Memory/context matching             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ v_proj       │ Value projection        │ "What info to pass forward?"        │
# │              │                         │ Content selection                   │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ o_proj       │ Output projection       │ Combines attention heads            │
# │              │                         │ Post-attention mixing               │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# MLP/FFN MODULES (More aggressive - use for knowledge injection)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ gate_proj    │ Gate projection         │ Controls information flow           │
# │              │                         │ Heavy knowledge modification        │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ up_proj      │ Up projection           │ Expands to hidden dimension         │
# │              │                         │ Feature extraction                  │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ down_proj    │ Down projection         │ Compresses back to model dim        │
# │              │                         │ Output generation                   │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# PRESETS:
#   Conservative (style/persona):  ["q_proj", "v_proj"]
#   Balanced:                      ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (new knowledge):    ["q_proj", "k_proj", "v_proj", "o_proj", 
#                                   "gate_proj", "up_proj", "down_proj"]
#
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Conservative: attention-only

# =========================== VLLM DEPLOYMENT ===========================
# Centralized vLLM models directory on A5000 server (mapped to /models in container)
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
# Test prompt for inference validation
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded (70B 4-bit QLoRA version)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Deployment: vLLM on A5000 GPU server")

✓ Configuration loaded (70B 4-bit QLoRA version)
  Base model: unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit
  Model name: biblical_llama31_70b_instruct_unsloth_4bit
  Input data: /home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset
  Output base: ./output/biblical_llama31_70b_instruct_unsloth_4bit
  LoRA config: r=8, alpha=16, targets=['q_proj', 'v_proj']
  Training: batch=1, grad_accum=8, lr=0.0001
  Training precision: 4-bit QLoRA
  Deployment: vLLM on A5000 GPU server


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [1]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmcompressor 0.9.0.1 requires torch<=2.9.1,>=2.7.0, but you have torch 2.10.0+cu130 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth: 2026.2.1
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [4]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")

📚 LOADING ALL AUGMENTOOLKIT DATA
Found 50 subdirectories
  Loaded 123 from factual_sft_full_biblical_data_followup_0/simplified_data_rag.jsonl
  Loaded 123 from factual_sft_full_biblical_data_followup_0/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/plain_qa_list.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/simplified_data_rag.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/plain_qa_list.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/simplified_data_rag.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/plain_qa_list.jsonl
  Loaded 112 from factual_sft_full_biblical_data_followup_5/simplified_data_rag.jsonl
  Loaded 112 from factual_s

Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Pippa-Thoughts_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Bluemoon-1mil-thoughts_0.jsonl: An error occurred while generating the dataset


...

  Skipped Augmentoolkit-Augmentoolkit-LMsys-800k-Thoughts_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Openthoughts-100mil-DifferentFormat_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Capybara-2point5mil-Thoughts_0.jsonl: An error occurred while generating the dataset
  Loaded 2309 from inferred_facts_full_biblical_data/final_output.jsonl
  Loaded 26 from pretraining_run/text_chunks_full_biblical_data.jsonl
  Loaded 2199 from pretraining_run/representation_variation_full_biblical_data.jsonl
  Loaded 2309 from pretraining_run/inferred_facts_full_biblical_data.jsonl
  Loaded 134 from rag_data_full_biblical_data/axolotl_rag_conversations.jsonl
  Loaded 5932 from rag_source_data/rag_data_full_biblical_data.jsonl
  Loaded 5932 from rag_source_data/rag_data_combined.jsonl
  Loaded 2199 from representation_variation_full_biblical_data/final_output.jsonl
  Loaded 26 from sft_run/pretraining_subset_441912.jsonl
  Loaded 134 from sft_run/axolotl_rag_conversations_full_biblical_data.jsonl

✓ Total dataset loaded: 30274 examples
✓ Columns: ['conversations', 'text', 'metadata', 'segments', 'question', 'source_text', 'source_metadata', 'relat

## 4. Load Model & Tokenizer with Unsloth (4-bit)

Load Llama 3.1 70B Instruct model in **4-bit precision** for QLoRA training.

**Training on DGX Spark (128GB unified memory):**
- 70B in 4-bit fits comfortably (~40GB)
- Plenty of headroom for optimizer states and activations

**Deployment on A5000 (24GB VRAM):**
- 4-bit merged model served via vLLM
- vLLM handles KV-cache management efficiently for 70B at 4-bit

In [7]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
# Use device_map={"": 0} to force all layers onto GPU 0
# DGX Spark has 119.7GB — plenty for 70B 4-bit (~40GB)
# device_map="auto" incorrectly offloads some layers to CPU which BnB 4-bit rejects
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map={"": 0}    # Force all layers onto GPU 0 (119GB unified memory)
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: 4-bit (QLoRA)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [03:28<00:00, 34.67s/it]


✓ Model loaded: unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit
✓ Precision: 4-bit (QLoRA)
✓ Tokenizer configured
✓ Max sequence length: 2048


In [8]:
# Format dataset for Llama 3.1 chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n✓ Dataset formatted: {len(dataset)} examples")

Filter: 100%|██████████| 30274/30274 [00:00<00:00, 783265.85 examples/s]


--- Sample formatted text (first 500 chars) ---
[[[OVERALL_CONTEXT_IS -> Biblical Faith and Apostolic Teaching]]]
Specific source: habakkuk.txt
The ultimate consequence of Habakkuk ' s vision is that the Lord God is his strength, andhe will make HahaukKk 's feet likehinds ' fSst, WnaGlinN him to walk upon his high places. This is the culmination of a series of events that began with Habakkuk ' scryto God, feeling that God was not hearing him. However, God had already set in motion a plan to worka work in the days of Habakkuk ' s audience that

✓ Dataset formatted: 18142 examples


## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [9]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 80 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA adapters added (r=8, targets=['q_proj', 'v_proj'])
✓ Trainable parameters: 16,384,000


## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (1e-4) to gently teach persona
- 70B is already highly capable — minimal fine-tuning is sufficient
- This preserves base model capabilities while adding authentic voice

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(100, steps_per_epoch)  # Save every 100 steps or at epoch end (more frequent for 70B)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,           # Log more frequently for 70B (fewer total steps)
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for 70B 4-bit QLoRA training")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps")
print(f"✓ Learning rate: {LEARNING_RATE}")

Unsloth: Tokenizing ["text"] (num_proc=24): 100%|██████████| 18142/18142 [00:05<00:00, 3462.33 examples/s]

✓ Trainer configured for 70B 4-bit QLoRA training
✓ Dataset size: 18142 conversations
✓ Effective batch size: 8 (batch=1 × grad_accum=8)
✓ Steps per epoch: 2267
✓ Total epochs: 1
✓ Total steps: 2267
✓ Warmup steps: 226
✓ Save every: 100 steps
✓ Learning rate: 0.0001


In [11]:
# Start training
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,142 | Num Epochs = 1 | Total steps = 2,267
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 16,384,000 of 70,570,090,496 (0.02% trained)


Step,Training Loss
5,1.128700
10,1.344500
15,1.486900
20,1.340800
25,1.350200
30,1.386600
35,1.399400
40,1.271500
45,1.309500
50,1.288900


KeyboardInterrupt: 

## 7. Save LoRA Adapters

Save the trained LoRA adapters separately. These can be loaded on any Llama 3.1 70B model later.

**This is the primary output** - merging to full model is optional (Step 8).

In [ ]:
from pathlib import Path

# Save LoRA adapters (primary output)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"💾 Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"✅ LoRA adapters saved!")
print(f"   Path: {LORA_OUTPUT_DIR}")
print(f"   Can be loaded on any Llama 3.1 70B model with PEFT")